<a href="https://colab.research.google.com/github/turkim456/IMA-AI-Home-Maintenance-Agent/blob/main/IMA_AI_Home_Maintenance_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U langchain langchain-openai langgraph gradio gspread pandas pydantic

In [ ]:
import os
import pandas as pd
from pydantic import BaseModel, Field

# 1. إعداد مفتاح الذكاء الاصطناعي (نموذج gpt-4o-mini)
os.environ["OPENROUTER_API_KEY"] = "OPENROUTER_API_KEY"

from langchain_openai import ChatOpenAI
model = ChatOpenAI(
    model="openai/gpt-4o-mini",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    temperature=0 # جعل درجة الحرارة 0 يضمن دقة عالية وتصنيفاً صارماً
)
print("✅ تم تفعيل نموذج الذكاء الاصطناعي.")

# 2. المصادقة مع قوقل شيت (سيطلب منك كولاب صلاحية الدخول لحسابك)
from google.colab import auth
from google.auth import default
import gspread

auth.authenticate_user()
credentials, _ = default()
gc = gspread.authorize(credentials)

try:
    # سيقوم بفتح ملف قوقل شيت (تأكد من إنشاء ملف بهذا الاسم في درايف)
    sheet = gc.open("IMA_Requests").sheet1
    print("✅ تم الاتصال بقوقل شيت بنجاح.")
except Exception as e:
    print("⚠️ لم يتم العثور على ملف قوقل شيت. الرجاء إنشاء ملف باسم IMA_Requests")

# 3. قراءة ملف الكتالوج (CSV)
try:
    df_catalog = pd.read_csv('/content/فئات رئيسية و فرعية و خدمات للصيانة.csv')
    print("✅ تم قراءة كتالوج الخدمات بنجاح. عدد الخدمات المتاحة:", len(df_catalog))
except FileNotFoundError:
    print("❌ الرجاء رفع ملف 'ممتاز_لنبدا_الان_بالفئات_الفرع.csv' إلى كولاب أولاً.")

# 4. هيكلة البيانات (Pydantic Schema)
# لاحظ كيف نستخدم أسماء إنجليزية للمتغيرات، ولكن نوجه النموذج بالعربي في الـ description
class MaintenanceRequestData(BaseModel):
    main_category: str = Field(description="الفئة الرئيسية للمشكلة (مثل: كهرباء، سباكة، تكييف)")
    sub_category: str = Field(description="الفئة الفرعية للمشكلة (مثل: الإنارة والإضاءة)")
    service_name: str = Field(description="اسم الخدمة الدقيق بناءً على الكتالوج")
    problem_description: str = Field(description="وصف المشكلة كما ذكرها العميل بالتفصيل")
    customer_name: str = Field(description="اسم العميل")
    city: str = Field(description="المدينة التي يقطن بها العميل")
    neighborhood: str = Field(description="الحي الذي يقطن به العميل")
    house_number: str = Field(description="رقم المنزل أو البناية")

✅ تم تفعيل نموذج الذكاء الاصطناعي.
✅ تم الاتصال بقوقل شيت بنجاح.
✅ تم قراءة كتالوج الخدمات بنجاح. عدد الخدمات المتاحة: 57


In [ ]:
from langchain_core.tools import tool
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage, AIMessage

# --- تحديث 1: تطوير أداة البحث ---
@tool
def search_maintenance_catalog(keywords: str) -> str:
    """
    أداة للبحث في كتالوج خدمات الصيانة.
    يجب تمرير "كلمة مفتاحية واحدة أو اثنتين فقط" (مثال: 'كهرباء'، 'طبلون'، 'إضاءة'، 'تسرب') وليس جملة كاملة.
    """
    # البحث بمرونة أكبر (البحث عن أي جزء من الكلمة المفتاحية)
    mask = df_catalog.apply(lambda row: row.astype(str).str.contains(keywords, case=False, na=False).any(), axis=1)
    results = df_catalog[mask]

    if results.empty:
        # إذا لم يجد نتيجة دقيقة، نرجع تصنيفاً عاماً ليتمكن الوكيل من الاستمرار
        return "لم يتم العثور على خدمة دقيقة. الرجاء تصنيفها كـ 'صيانة عامة' واستكمال أخذ بيانات العميل (الاسم، الحي، الخ)."

    top_results = results.head(3)
    response = "الخدمات المقترحة:\n"
    for _, row in top_results.iterrows():
        response += f"- الفئة: {row['الفئة الرئيسية']} | الخدمة: {row['اسم الخدمة']}\n"
    return response

@tool
def save_to_google_sheet(main_category: str, sub_category: str, service_name: str, problem_description: str, customer_name: str, city: str, neighborhood: str, house_number: str) -> str:
    """
    تحفظ بيانات الطلب في قوقل شيت. لا تستخدمها إلا بعد جمع كل البيانات الثمانية.
    """
    try:
        row = [main_category, sub_category, service_name, problem_description, customer_name, city, neighborhood, house_number]
        sheet.append_row(row)
        return "تم حفظ الطلب بنجاح في قاعدة البيانات."
    except Exception as e:
        return f"حدث خطأ أثناء الحفظ: {str(e)}"

tools = [search_maintenance_catalog, save_to_google_sheet]
model_with_tools = model.bind_tools(tools)

# --- تحديث 2: هندسة الحالة والتعليمات لمنع التكرار ---
class MaintenanceState(TypedDict):
    messages: Annotated[list, add_messages]

# أزلنا عقدة validate_input لأنها كانت تسبب تكرار عبارة "التفاصيل غير كافية"
# وسنعتمد على ذكاء النموذج في الـ System Prompt لإدارة الحوار.

system_prompt = """أنت "Interactive Maintenance Assistant (IMA)"، وكيل ذكاء اصطناعي تفاعلي ".
تتحدث العربية دائماً بأسلوب لبق.

مسار عملك الصارم:
1. استمع لمشكلة العميل.
2. استخدم أداة 'search_maintenance_catalog' *مرة واحدة فقط* بكلمات مفتاحية قصيرة جداً (مثل: 'كهرباء'، 'انقطاع') لمعرفة التصنيف.
3. إذا أرجعت الأداة نتائج، اختر الأنسب. وإذا لم تجد، صنفها كـ 'صيانة عامة'. لا تخبر العميل أنك "لم تعثر على الخدمة"، بل قل "فهمت مشكلتك".
4. ابدأ فوراً بجمع البيانات المفقودة (اسم العميل، المدينة، الحي، رقم المنزل).
5. القاعدة الذهبية: اسأل سؤالاً واحداً فقط في كل رد (مثال: "ما هو اسمك الكريم؟").
6. لا تكرر إجاباتك السابقة.
7. عندما تكتمل جميع البيانات، استدعِ أداة 'save_to_google_sheet' لحفظ الطلب، ثم اشكر العميل وأنهِ المحادثة.
"""

def agent_node(state: MaintenanceState) -> MaintenanceState:
    messages = [SystemMessage(content=system_prompt)] + state["messages"]
    response = model_with_tools.invoke(messages)
    return {"messages": [response]}

builder = StateGraph(MaintenanceState)
builder.add_node("agent", agent_node)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "agent")
builder.add_conditional_edges(
    "agent",
    tools_condition,
    {"tools": "tools", "__end__": END}
)
builder.add_edge("tools", "agent")

memory = InMemorySaver()
ima_agent = builder.compile(checkpointer=memory)

print("✅ تم تحديث الوكيل: تم تحسين دقة البحث ومنع التكرار المزعج!")

✅ تم تحديث الوكيل: تم تحسين دقة البحث ومنع التكرار المزعج!


In [ ]:
import gradio as gr
import uuid

# إنشاء رقم جلسة فريد لكل مستخدم لضمان عمل الذاكرة بشكل مستقل
thread_id = str(uuid.uuid4())

def chat_with_ima(user_input, chat_history):
    if not user_input.strip():
        return "", chat_history, {"status": "يرجى إدخال نص صحيح."}

    # إعداد الذاكرة للجلسة الحالية
    config = {"configurable": {"thread_id": thread_id}}

    # تشغيل الوكيل (LangGraph)
    result = ima_agent.invoke(
        {"messages": [{"role": "user", "content": user_input}]},
        config=config
    )

    # استخراج رد الوكيل
    agent_response = result["messages"][-1].content

    # الحل هنا: تحديث المحادثة لتتوافق مع صيغة Gradio الجديدة (messages)
    chat_history.append({"role": "user", "content": user_input})
    chat_history.append({"role": "assistant", "content": agent_response})

    # البحث عن البيانات المهيكلة لعرضها في شاشة مزود الخدمة
    dashboard_data = {"status": "جاري جمع البيانات من العميل..."}
    for msg in result["messages"]:
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            for tool_call in msg.tool_calls:
                if tool_call['name'] == 'save_to_google_sheet':
                    dashboard_data = tool_call['args']
                    dashboard_data["System_Status"] = "✅ تم إرسال الطلب بنجاح إلى قوقل شيت"

    return "", chat_history, dashboard_data
# مسح المحادثة لبدء سيناريو جديد
def clear_chat():
    global thread_id
    thread_id = str(uuid.uuid4()) # توليد جلسة جديدة بذاكرة فارغة
    return [], {"status": "بانتظار طلب جديد..."}

# بناء الواجهة (Gradio Blocks)
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        <div style="text-align: center;">
            <h1>🛠️ Interactive Maintenance Assistant (IMA)</h1>
            <h3>الوكيل الذكي لتصنيف طلبات الصيانة المنزلية</h3>
        </div>
        """
    )

    with gr.Row():
        # القسم الأول: واجهة العميل
        with gr.Column(scale=2):
            gr.Markdown("### 📱 واجهة العميل")
            chatbot = gr.Chatbot(label="المحادثة مع الوكيل", height=400)
            user_input = gr.Textbox(
                label="رسالتك",
                placeholder="اكتب طلب الصيانة هنا... (مثال: عندي التماس في الكهرباء شمال الرياض)"
            )
            with gr.Row():
                send_btn = gr.Button("إرسال الطلب", variant="primary")
                clear_btn = gr.Button("بدء محادثة جديدة")

        # القسم الثاني: واجهة مزود الخدمة (لوحة التحكم)
        with gr.Column(scale=1):
            gr.Markdown("### 💻 واجهة مزود الخدمة (البيانات المهيكلة)")
            json_output = gr.JSON(label="تفاصيل الطلب (JSON)")

    # ربط الأزرار بالدوال البرمجية
    send_btn.click(
        chat_with_ima,
        inputs=[user_input, chatbot],
        outputs=[user_input, chatbot, json_output]
    )
    user_input.submit(
        chat_with_ima,
        inputs=[user_input, chatbot],
        outputs=[user_input, chatbot, json_output]
    )
    clear_btn.click(
        clear_chat,
        inputs=[],
        outputs=[chatbot, json_output]
    )

# تشغيل التطبيق وإصدار الرابط العام
demo.launch(debug=True, share=True)

/tmp/ipykernel_39091/1223660179.py:44: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://92ee4c652ff13e3569.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
